# Sodetlib Config Overview

These commands make sure that matplotlib is configured correctly to run in 
docker. It makes matplotlib use a non-graphical backend, but allows you to 
still show plots inline in a notebook. This bypasses docker graphical issues
that usually pop up, and will be needed in most notebooks.

In [12]:
import matplotlib
matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plt

In [4]:
from sodetlib.det_config import DetConfig
from pprint import pprint

The DetConfig object is used to load system, device, and pysmurf configurations. By default, it will first load the system configuration from `$OCS_CONFIG_DIR/sys_config.yml`. This config file should contain the device and pysmurf config files to use for each smurf slot, which the DetConfig system will then load.

Here I am creating a cfg object and loading in the configurations. The `parse_args` function takes in an optional argument list, and parser, and it will parse either the command line args, or the args list if specified with the supplied parser and a few extra arguments. If more than one slot is present on the system, the `--slot` (or `-N`) variable must be set to the slot for which to load the config. The `--sys-file`, `--dev-file`, and `--pysmurf-file` arguments can also be set to specify which config files to use manually.

In [5]:
cfg = DetConfig()
cfg.load_config_files(slot=2)

SO config data is split into system config, and device config, stored in `cfg.sys` and `cfg.dev` respectively.

In [6]:
print("Sys config\n" + 13*"-")
pprint(cfg.sys)

Sys config
-------------
{'comm_type': 'eth',
 'crate_id': 1,
 'docker_env': {'ATCA_MONITOR_TAG': 'R1.0.0',
                'CB_HOST': '192.168.0.120',
                'PYSMURF_CLIENT_TAG': 'v4.1.0',
                'SOCS_TAG': 'v0.1.0-3-gaf8873a-dev',
                'SODETLIB_TAG': 'v0.0.1',
                'STREAMER_TAG': 'v0.0.2-v2.1.0-stable'},
 'max_fan_level': 10,
 'meta_register_file': '$OCS_CONFIG_DIR/meta_registers.yaml',
 'shelf_manager': 'shm-smrf-sp01',
 'slot_order': [2],
 'slots': {'SLOT[2]': {'device_config': '$OCS_CONFIG_DIR/device_configs/dev_cfg_demo_test_s2.yaml',
                       'pysmurf_config': '/config/pysmurf_config/experiment_ucsd_k2so_cc02-06_lbOnlyBay0.cfg',
                       'stream_port': 4532}},
 'startup': {'configure_pysmurf': True,
             'reboot': True,
             'run_half_band_test': False,
             'set_crate_fans_to_full': True,
             'start_atca_monitor': False,
             'write_config': False}}


Device config is split into Experiment config, or `exp`, which contains general config info about the device,
`bands` which contains info about the 8 bands on the slot, and `bias_groups` which contains config info about the 12 bias groups.

In [7]:
print("Exp\n"+5*"-")
pprint(cfg.dev.exp)
print("\nBand[0]\n" + 10*'-')
pprint(cfg.dev.bands[3])
print("\nBiasGroup[0]\n" + 10*'-')
pprint(cfg.dev.bias_groups[0])

Exp
-----
{'amp_50k_Id': 14,
 'amp_50k_Vg': -0.5179895424000001,
 'amp_hemt_Id': 8,
 'amp_hemt_Vg': -0.8299942848,
 'tunefile': '/data/smurf_data/tune/1606763812_tune.npy'}

Band[0]
----------
{'active_subbands': [12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     19,
                     20,
                     30,
                     31,
                     32,
                     33,
                     34,
                     35,
                     36,
                     37,
                     38,
                     39,
                     40,
                     41,
                     42,
                     43],
 'dc_att': 0,
 'detectors': [],
 'drive': 12,
 'feedback_end_frac': 0.98,
 'feedback_start_frac': 0.02,
 'flux_ramp_rate_khz': 4,
 'frac_pp': 0.2797240558589269,
 'lms_freq_hz': 20000.0,
 'lms_gain': 5,
 'nphi0': 4,
 'optimized_dri

## Creating pysmurf instance

To create a pysmurf instance based on this configuration setup, simply run: 

In [8]:
S = cfg.get_smurf_control(dump_configs=True)

This will create a control object with the correct epics root, pysmurf config file, etc. and dump all of the config files to S.data_directory. Using this will also correctly set the pysmurf publisher ID correctly based on the crate id and slot number.

# Running sodetlib functions

Sodetlib functions can be imported from the package `sodetlib.smurf_funcs`. For instance, if you want to run the health check script, you can:

In [10]:
from sodetlib.smurf_funcs import health_check

In [11]:
bay0, bay1 = True, False
health_check(S, cfg)


Checking biases
[ 2020-11-30 23:28:36 ]  {'hemt_Vg': -0.8299942848, 'hemt_Id': 7.586132787499999, '50K_Vg': -0.5179895424000001, '50K_Id': 13.53515625}

Scanning hemt bias voltage
[ 2020-11-30 23:28:36 ]  {'hemt_Vg': -0.8299942848, 'hemt_Id': 7.5990234124999985, '50K_Vg': -0.5179895424000001, '50K_Id': 13.53515625}

Scanning 50K bias voltage
[ 2020-11-30 23:28:36 ]  {'hemt_Vg': -0.8299942848, 'hemt_Id': 7.592578099999999, '50K_Vg': -0.5179895424000001, '50K_Id': 13.53515625}
[ 2020-11-30 23:28:36 ]  {'hemt_Vg': -0.8299942848, 'hemt_Id': 7.5990234124999985, '50K_Vg': -0.5179895424000001, '50K_Id': 13.53515625}
Final hemt current = 7.5990234124999985
Desired hemt current = 8
hemt current within range of desired value:  True
Final hemt gate voltage is -0.8299942848
Final 50K current = 13.53515625
Desired 50K current = 14
50K current within range of desired value:True
Final 50K gate voltage is -0.5179895424000001

Checking JESD Connections
[ 2020-11-30 23:28:36 ]  JESD Tx Okay
[ 2020-11-3

/usr/local/lib/python3.6/dist-packages/scipy/signal/spectral.py:1812: UserWarning: Input data is complex, switching to return_onesided=False
  warnings.warn('Input data is complex, switching to '


Full band response check passed

Health check finished! Final status
True - Hemt biased: 	True
 - Hemt Id in range: 	True
 - Hemt (Id, Vg): 	(7.5990234124999985, -0.8299942848)

True - 50K biased: 		True
 - 50K Id in range: 	True
 - 50K (Id, Vg): 	(13.53515625, -0.5179895424000001)

 - Response check: 	True
 - JESD[0] TX, RX: 	(True, True)


True